# DDRI 대표 15개 상위 5개 오류 스테이션 시간대 패턴 분석

- 목적: 대표 15개 `station-hour(대여소-시간 단위)` 실험의 상위 오류 5개 스테이션에 대해 시간대별 실제값-예측값 차이를 재생성한다.
- 범위: `2377`, `2348`, `4917`, `2328`, `2359`
- 입력:
  - `rep15_error_analysis/output/data/ddri_rep15_station_error_priority_table.csv`
  - `3조 공유폴더/대표대여소_예측데이터_15개/raw_data/ddri_prediction_long_train_2023_2024.csv`
  - `3조 공유폴더/대표대여소_예측데이터_15개/raw_data/ddri_prediction_long_test_2025.csv`
- 출력:
  - `rep15_error_analysis/output/data/ddri_rep15_top5_station_hourly_error_patterns.csv`
  - `rep15_error_analysis/output/data/ddri_rep15_top5_station_error_summary.csv`
  - `rep15_error_analysis/output/data/ddri_rep15_top5_station_peak_error_hours.csv`


In [1]:
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
ANALYSIS_DIR = ROOT / 'works/05_prediction_long/cheng80/rep15_error_analysis'
OUTPUT_DATA_DIR = ANALYSIS_DIR / 'output/data'
RAW_DATA_DIR = ROOT / '3조 공유폴더/대표대여소_예측데이터_15개/raw_data'
PRIORITY_PATH = OUTPUT_DATA_DIR / 'ddri_rep15_station_error_priority_table.csv'
TRAIN_PATH = RAW_DATA_DIR / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = RAW_DATA_DIR / 'ddri_prediction_long_test_2025.csv'

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
priority_df = pd.read_csv(PRIORITY_PATH, encoding='utf-8-sig')
priority_df['station_id'] = priority_df['station_id'].astype(str)
top5_ids = priority_df.head(5)['station_id'].tolist()

train_df = pd.read_csv(TRAIN_PATH, encoding='utf-8-sig')
test_df = pd.read_csv(TEST_PATH, encoding='utf-8-sig')

feature_cols = [
    'station_id', 'station_name', 'station_group', 'hour', 'cluster', 'mapped_dong_code',
    'weekday', 'month', 'holiday', 'temperature', 'humidity', 'precipitation', 'wind_speed'
]
cat_cols = ['station_id', 'station_name', 'station_group', 'cluster', 'mapped_dong_code', 'weekday', 'month', 'holiday', 'hour']

X_train = train_df[feature_cols].copy()
y_train = train_df['rental_count'].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df['rental_count'].copy()

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)
model.fit(X_train, y_train, categorical_feature=cat_cols)

pred = model.predict(X_test)

test_eval = test_df[['station_id', 'station_name', 'station_group', 'date', 'hour', 'rental_count']].copy()
test_eval['station_id'] = test_eval['station_id'].astype(str)
test_eval['prediction'] = pred
test_eval['residual'] = test_eval['rental_count'] - test_eval['prediction']
test_eval['abs_error'] = test_eval['residual'].abs()
test_eval['squared_error'] = test_eval['residual'] ** 2

target_eval = test_eval[test_eval['station_id'].isin(top5_ids)].copy()
target_eval['station_id'] = pd.Categorical(target_eval['station_id'], categories=top5_ids, ordered=True)

station_summary = target_eval.groupby(['station_id', 'station_name', 'station_group'], as_index=False, observed=True).agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: (s.mean()) ** 0.5),
    bias_mean=('residual', 'mean'),
)
station_summary['analysis_rank'] = range(1, len(station_summary) + 1)
station_summary = station_summary[['analysis_rank', 'station_id', 'station_name', 'station_group', 'actual_mean', 'predicted_mean', 'mae', 'rmse', 'bias_mean']]

hourly_patterns = target_eval.groupby(['station_id', 'station_name', 'station_group', 'hour'], as_index=False, observed=True).agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: (s.mean()) ** 0.5),
    gap_mean=('residual', 'mean'),
).sort_values(['station_id', 'rmse', 'mae', 'hour'], ascending=[True, False, False, True])

peak_hours = hourly_patterns.groupby('station_id', group_keys=False, observed=True).head(5).copy()
peak_hours['peak_rank'] = peak_hours.groupby('station_id', observed=True).cumcount() + 1
peak_hours = peak_hours[['station_id', 'station_name', 'station_group', 'peak_rank', 'hour', 'actual_mean', 'predicted_mean', 'mae', 'rmse', 'gap_mean']]

station_summary_path = OUTPUT_DATA_DIR / 'ddri_rep15_top5_station_error_summary.csv'
hourly_patterns_path = OUTPUT_DATA_DIR / 'ddri_rep15_top5_station_hourly_error_patterns.csv'
peak_hours_path = OUTPUT_DATA_DIR / 'ddri_rep15_top5_station_peak_error_hours.csv'

station_summary.to_csv(station_summary_path, index=False, encoding='utf-8-sig')
hourly_patterns.to_csv(hourly_patterns_path, index=False, encoding='utf-8-sig')
peak_hours.to_csv(peak_hours_path, index=False, encoding='utf-8-sig')

print(f'saved: {station_summary_path}')
print(f'saved: {hourly_patterns_path}')
print(f'saved: {peak_hours_path}')


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002281 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 813
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 13
[LightGBM] [Info] Start training from score 0.724514


saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/cheng80/rep15_error_analysis/output/data/ddri_rep15_top5_station_error_summary.csv
saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/cheng80/rep15_error_analysis/output/data/ddri_rep15_top5_station_hourly_error_patterns.csv
saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/cheng80/rep15_error_analysis/output/data/ddri_rep15_top5_station_peak_error_hours.csv


In [3]:
station_summary


,analysis_rank,station_id,station_name,station_group,actual_mean,predicted_mean,mae,rmse,bias_mean
0,1,2377,수서역 5번출구,아침 도착 업무 집중형,1.822603,1.930406,1.184462,1.716840,-0.107803
1,2,2348,포스코사거리(기업은행),아침 도착 업무 집중형,1.475228,1.861062,1.053046,1.690407,-0.385833
2,3,4917,일원에코파크 주차장,주거 도착형,1.104452,0.804239,0.834525,1.275094,0.300213
3,4,2328,르네상스 호텔 사거리 역삼지하보도 7번출구 앞,업무/상업 혼합형,0.649429,0.762303,0.671325,0.946361,-0.112874
4,5,2359,"국립국악중,고교 정문 맞은편",외곽 주거형,0.762100,0.789788,0.668261,0.939339,-0.027688


In [4]:
peak_hours.sort_values(['station_id', 'peak_rank'])


,station_id,station_name,station_group,peak_rank,hour,actual_mean,predicted_mean,mae,rmse,gap_mean
18,2377,수서역 5번출구,아침 도착 업무 집중형,1,18,5.852055,6.008468,2.602583,3.413188,-0.156413
19,2377,수서역 5번출구,아침 도착 업무 집중형,2,19,3.923288,3.875234,2.382748,3.158294,0.048054
17,2377,수서역 5번출구,아침 도착 업무 집중형,3,17,4.169863,4.602516,2.045808,2.633699,-0.432653
20,2377,수서역 5번출구,아침 도착 업무 집중형,4,20,2.657534,2.645472,1.673139,2.181969,0.012062
16,2377,수서역 5번출구,아침 도착 업무 집중형,5,16,2.969863,3.068608,1.643067,2.132021,-0.098745
42,2348,포스코사거리(기업은행),아침 도착 업무 집중형,1,18,6.939726,8.006162,3.392415,4.769775,-1.066436
41,2348,포스코사거리(기업은행),아침 도착 업무 집중형,2,17,5.676712,7.495691,2.615563,3.560949,-1.818978
40,2348,포스코사거리(기업은행),아침 도착 업무 집중형,3,16,2.827397,3.539672,1.534984,1.977867,-0.712275
43,2348,포스코사거리(기업은행),아침 도착 업무 집중형,4,19,2.169863,2.680857,1.527397,1.932763,-0.510994
35,2348,포스코사거리(기업은행),아침 도착 업무 집중형,5,11,2.079452,2.272217,1.244145,1.794485,-0.192765


## 해석 메모

- `2377`, `2348`는 여전히 `16~19시` 저녁 피크 시간대가 핵심 오류 구간이다.
- `4917`은 `05~08시` 아침 시간대 과소예측이 두드러진다.
- `2328`은 `17~18시`와 `08시`, `11시`가 주요 확인 시간대다.
- `2359`는 `08시`, `16~18시`, `20시`가 상위 오류 시간대다.
- 다음 단계는 이 5개 스테이션을 군집별 후보 피처와 직접 연결해 subset(축소 피처 조합) 실험 우선순위를 정하는 것이다.
